In [1]:
from google.colab import files
files.upload()

Saving kaggle.json to kaggle.json


{'kaggle.json': b'{"username":"dominika2024","key":"5b70a7c216956767dcbe2e100185e54d"}'}

In [2]:
import os
import shutil

os.makedirs('/root/.kaggle', exist_ok=True)
shutil.move('kaggle.json', '/root/.kaggle/kaggle.json')
os.chmod('/root/.kaggle/kaggle.json', 600)

In [3]:
!kaggle competitions download -c multi-view-pig-posture-recognition

100% 1.82G/1.82G [00:50<00:00, 38.6MB/s]



In [4]:
!unzip multi-view-pig-posture-recognition.zip -d data

Streaming output truncated to the last 5000 lines.
  inflating: data/multiview_pig_posture_recognition/train1_images/pen1_tur_cam2_20250207_135939.jpg  
  inflating: data/multiview_pig_posture_recognition/train1_images/pen1_tur_cam2_20250207_140259.jpg  
  inflating: data/multiview_pig_posture_recognition/train1_images/pen1_tur_cam2_20250207_140439.jpg  
  inflating: data/multiview_pig_posture_recognition/train1_images/pen1_tur_cam2_20250207_140759.jpg  
  inflating: data/multiview_pig_posture_recognition/train1_images/pen1_tur_cam2_20250207_140940.jpg  
  inflating: data/multiview_pig_posture_recognition/train1_images/pen1_tur_cam2_20250207_141300.jpg  
  inflating: data/multiview_pig_posture_recognition/train1_images/pen1_tur_cam2_20250207_141439.jpg  
  inflating: data/multiview_pig_posture_recognition/train1_images/pen1_tur_cam2_20250207_141759.jpg  
  inflating: data/multiview_pig_posture_recognition/train1_images/pen1_tur_cam2_20250207_141940.jpg  
  inflating: data/multiview_pig

In [ ]:
!ls data/multiview_pig_posture_recognition

In [5]:
import os
import ast
import copy
import numpy as np
import pandas as pd
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler

from torchvision import transforms, models

from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix

from tqdm.auto import tqdm
import matplotlib.pyplot as plt

In [6]:
CLASS_NAMES = {
    0: "Lateral_lying_left",
    1: "Lateral_lying_right",
    2: "Sitting",
    3: "Standing",
    4: "Sternal_lying",
}

BASE_DIR = "data/multiview_pig_posture_recognition"

TRAIN2_CSV = os.path.join(BASE_DIR, "train2.csv")
TEST_CSV   = os.path.join(BASE_DIR, "test.csv")

train2 = pd.read_csv(TRAIN2_CSV)
test   = pd.read_csv(TEST_CSV)

train = train2.copy()
train["source"] = "train2"
test["source"]  = "test"

train["bbox_parsed"] = train["bbox"].apply(ast.literal_eval)
test["bbox_parsed"]  = test["bbox"].apply(ast.literal_eval)

train["class_name"] = train["class_id"].map(CLASS_NAMES)

print("Train rows:",  len(train))
print("Train images:", train["image_id"].nunique())
print(train["class_name"].value_counts())
print("Test rows:",   len(test))

Train rows: 23450
Train images: 3150
class_name
Standing               9928
Sternal_lying          6309
Lateral_lying_right    3435
Lateral_lying_left     3083
Sitting                 695
Name: count, dtype: int64
Test rows: 11708


In [7]:
def crop_with_padding(image, bbox, padding=0.12, make_square=True):
    img_w, img_h = image.size
    x, y, w, h = map(float, bbox)

    pad_x = w * padding
    pad_y = h * padding

    x1 = x - pad_x
    y1 = y - pad_y
    x2 = x + w + pad_x
    y2 = y + h + pad_y

    if make_square:
        crop_w = x2 - x1
        crop_h = y2 - y1
        side = max(crop_w, crop_h)
        cx = (x1 + x2) / 2
        cy = (y1 + y2) / 2
        x1 = cx - side / 2
        x2 = cx + side / 2
        y1 = cy - side / 2
        y2 = cy + side / 2

    x1 = max(0, int(round(x1)))
    y1 = max(0, int(round(y1)))
    x2 = min(img_w, int(round(x2)))
    y2 = min(img_h, int(round(y2)))

    if x2 <= x1: x2 = min(img_w, x1 + 1)
    if y2 <= y1: y2 = min(img_h, y1 + 1)

    return image.crop((x1, y1, x2, y2))


def load_image(image_id, source):
    folder_map = {
        "train1": "train1_images",
        "train2": "train2_images",
        "test":   "test_images",
    }
    if source not in folder_map:
        raise ValueError(f"Unknown source: {source}")
    path = os.path.join(BASE_DIR, folder_map[source], image_id)
    return Image.open(path).convert("RGB")

In [8]:
class PigPostureDataset(Dataset):
    """Standard supervised dataset (train2 + inference)."""
    def __init__(self, df, transform=None, is_test=False):
        self.df = df.reset_index(drop=True)
        self.transform = transform
        self.is_test = is_test

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        image = load_image(row["image_id"], row["source"])
        crop  = crop_with_padding(image, row["bbox_parsed"], padding=0.12, make_square=True)
        if self.transform:
            crop = self.transform(crop)
        if self.is_test:
            return crop, row["row_id"]
        return crop, int(row["class_id"])

## SSL Pre-training — Rotation Prediction

Trenujemy model **bez etykiet posture** na zadaniu przewidywania rotacji obrazu (0°, 90°, 180°, 270°).  
Używamy **train2 + cały test set** — dzięki temu model "widzi" kamery testowe jeszcze przed właściwym treningiem.  
To pomaga rozróżniać `Lateral_lying_left` vs `Lateral_lying_right`, bo model uczy się rozumieć orientację przestrzenną świni.

In [9]:
class RotationDataset(Dataset):
    """
    Pretext task: przewiduj o ile obrócono crop (0/90/180/270 stopni).
    Działa na train2 + test bez potrzeby etykiet posture.
    """
    ANGLES = [0, 90, 180, 270]

    def __init__(self, df, transform=None):
        self.df = df.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.df) * 4  # każdy crop x4 rotacje

    def __getitem__(self, idx):
        row_idx   = idx // 4
        angle_idx = idx % 4

        row   = self.df.iloc[row_idx]
        image = load_image(row["image_id"], row["source"])
        crop  = crop_with_padding(image, row["bbox_parsed"], padding=0.12, make_square=True)

        # Obróć PIL image
        crop = crop.rotate(self.ANGLES[angle_idx], expand=False)

        if self.transform:
            crop = self.transform(crop)

        return crop, angle_idx  # label = 0,1,2,3

In [10]:
class AddGaussianNoise:
    def __init__(self, mean=0.0, std=0.05, p=0.3):
        self.mean = mean
        self.std  = std
        self.p    = p

    def __call__(self, tensor):
        if torch.rand(1).item() < self.p:
            tensor = tensor + torch.randn_like(tensor) * self.std + self.mean
            tensor = torch.clamp(tensor, 0.0, 1.0)
        return tensor


# Transform do SSL (bez agresywnych rotacji — rotacje robimy ręcznie w datasecie)
ssl_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.2, hue=0.05),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

# Transform do właściwego treningu — mocny augmentation dla domain shift
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.3),
    transforms.RandomRotation(degrees=90),      # mocniej niż wcześniej
    transforms.RandomAffine(degrees=0, translate=(0.08, 0.08), scale=(0.9, 1.1)),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.1, hue=0.03),
    transforms.ToTensor(),
    AddGaussianNoise(p=0.2, std=0.03),
    transforms.RandomErasing(p=0.2, scale=(0.01, 0.05), ratio=(0.75, 1.33), value="random"),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

In [11]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

Device: cuda


## Faza 1 — SSL Pre-training na rotacjach (train2 + test)

In [15]:
# Łączymy train2 + test (bez etykiet posture, tylko cropujemy i obracamy)
ssl_df = test[["image_id", "source", "bbox_parsed"]].copy()

print(f"SSL dataset: {len(ssl_df)} cropów x4 rotacje = {len(ssl_df)*4} próbek")

ssl_dataset = RotationDataset(ssl_df, transform=ssl_transform)
ssl_loader  = DataLoader(ssl_dataset, batch_size=64, shuffle=True, num_workers=2, pin_memory=True)

SSL dataset: 11708 cropów x4 rotacje = 46832 próbek


In [16]:
# Model z głowicą na 4 klasy (rotacje)
ssl_model = models.efficientnet_v2_s(weights=models.EfficientNet_V2_S_Weights.IMAGENET1K_V1)
in_features = ssl_model.classifier[1].in_features
ssl_model.classifier[1] = nn.Linear(in_features, 4)
ssl_model = ssl_model.to(device)

optimizer_ssl = torch.optim.Adam(ssl_model.parameters(), lr=1e-4)
criterion_ssl = nn.CrossEntropyLoss()
scheduler_ssl = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer_ssl, T_max=5)

SSL_EPOCHS = 5
print(f"Startuję SSL pre-training ({SSL_EPOCHS} epok)...")

Startuję SSL pre-training (5 epok)...


In [17]:
for epoch in range(SSL_EPOCHS):
    ssl_model.train()
    total_loss = 0
    correct = 0
    total   = 0

    for images, labels in tqdm(ssl_loader, desc=f"SSL Epoch {epoch+1}/{SSL_EPOCHS}"):
        images = images.to(device)
        labels = labels.to(device)

        optimizer_ssl.zero_grad()
        outputs = ssl_model(images)
        loss    = criterion_ssl(outputs, labels)
        loss.backward()
        optimizer_ssl.step()

        total_loss += loss.item() * images.size(0)
        correct    += (outputs.argmax(1) == labels).sum().item()
        total      += images.size(0)

    scheduler_ssl.step()
    avg_loss = total_loss / total
    acc      = correct / total
    print(f"SSL Epoch {epoch+1}/{SSL_EPOCHS} | loss {avg_loss:.4f} | acc {acc:.4f}")

print("SSL pre-training gotowy!")

SSL Epoch 1/5:   0%|          | 0/732 [00:00<?, ?it/s]

SSL Epoch 1/5 | loss 0.1318 | acc 0.9506


SSL Epoch 2/5:   0%|          | 0/732 [00:00<?, ?it/s]

SSL Epoch 2/5 | loss 0.0059 | acc 0.9982


SSL Epoch 3/5:   0%|          | 0/732 [00:00<?, ?it/s]

SSL Epoch 3/5 | loss 0.0034 | acc 0.9992


SSL Epoch 4/5:   0%|          | 0/732 [00:00<?, ?it/s]

SSL Epoch 4/5 | loss 0.0010 | acc 0.9997


SSL Epoch 5/5:   0%|          | 0/732 [00:00<?, ?it/s]

SSL Epoch 5/5 | loss 0.0009 | acc 0.9998
SSL pre-training gotowy!


In [18]:
# Zapisz wagi backbone po SSL — przed fine-tuningiem
torch.save(ssl_model.state_dict(), "ssl_pretrained_backbone.pt")

from google.colab import files
files.download("ssl_pretrained_backbone.pt")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Faza 2 — Fine-tuning na train2 z etykietami posture

Przenosimy wagi backbone z SSL, zamieniamy głowicę na 5 klas posture i trenujemy normalnie.

In [19]:
# Przenosimy backbone z SSL → zamiana głowicy na 5 klas posture
ssl_model.classifier[1] = nn.Linear(in_features, len(CLASS_NAMES))
model = ssl_model.to(device)

print("Backbone przeniesiony z SSL. Głowica zamieniona na", len(CLASS_NAMES), "klas.")

Backbone przeniesiony z SSL. Głowica zamieniona na 5 klas.


In [20]:
# Cały train2 jako zbiór treningowy
train_df = train.reset_index(drop=True)
print("Train rows:",   len(train_df))
print("Train images:", train_df["image_id"].nunique())

Train rows: 23450
Train images: 3150


In [21]:
# WeightedRandomSampler — balansowanie klas
train_targets = train_df["class_id"].values
class_counts  = np.bincount(train_targets, minlength=len(CLASS_NAMES))
print("Class counts:", class_counts)

class_weights  = 1.0 / np.maximum(class_counts, 1)
sample_weights = np.array([class_weights[y] for y in train_targets])

sampler = WeightedRandomSampler(
    weights=torch.DoubleTensor(sample_weights),
    num_samples=len(sample_weights),
    replacement=True,
)

Class counts: [3083 3435  695 9928 6309]


In [22]:
BATCH_SIZE = 32

train_dataset = PigPostureDataset(train_df, transform=train_transform)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    sampler=sampler,
    num_workers=2,
    pin_memory=True,
)

# Sanity check
images, labels = next(iter(train_loader))
print("Batch shape:", images.shape)
print("Labels sample:", labels[:10].tolist())

Batch shape: torch.Size([32, 3, 224, 224])
Labels sample: [3, 3, 0, 0, 1, 4, 1, 1, 2, 4]


In [23]:
criterion  = nn.CrossEntropyLoss()
optimizer  = torch.optim.Adam(model.parameters(), lr=1e-4)
scheduler  = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=8)

In [24]:
def train_one_epoch(model, loader):
    model.train()
    total_loss = 0
    all_preds  = []
    all_labels = []

    for images, labels in tqdm(loader, leave=False):
        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss    = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * images.size(0)
        preds = outputs.argmax(dim=1)
        all_preds.extend(preds.detach().cpu().numpy())
        all_labels.extend(labels.detach().cpu().numpy())

    avg_loss = total_loss / len(loader.dataset)
    acc = accuracy_score(all_labels, all_preds)
    f1  = f1_score(all_labels, all_preds, average="macro")
    return avg_loss, acc, f1

In [25]:
EPOCHS  = 8
history = []

for epoch in range(EPOCHS):
    train_loss, train_acc, train_f1 = train_one_epoch(model, train_loader)
    scheduler.step()

    history.append({
        "epoch":      epoch + 1,
        "train_loss": train_loss,
        "train_acc":  train_acc,
        "train_f1":   train_f1,
    })

    print(
        f"Epoch {epoch+1}/{EPOCHS} | "
        f"loss {train_loss:.4f} | acc {train_acc:.4f} | f1 {train_f1:.4f}"
    )

torch.save({
    "model_state_dict": model.state_dict(),
    "class_names":      CLASS_NAMES,
}, "best_efficientnetv2_s_pigs.pt")

print("\nModel zapisany: best_efficientnetv2_s_pigs.pt")

  0%|          | 0/733 [00:00<?, ?it/s]

Epoch 1/8 | loss 0.7633 | acc 0.6901 | f1 0.6882


  0%|          | 0/733 [00:00<?, ?it/s]

Epoch 2/8 | loss 0.3536 | acc 0.8714 | f1 0.8706


  0%|          | 0/733 [00:00<?, ?it/s]

...

Epoch 6/8 | loss 0.1218 | acc 0.9588 | f1 0.9584


  0%|          | 0/733 [00:00<?, ?it/s]

Epoch 7/8 | loss 0.1023 | acc 0.9633 | f1 0.9636


  0%|          | 0/733 [00:00<?, ?it/s]

Epoch 8/8 | loss 0.0880 | acc 0.9696 | f1 0.9695

Model zapisany: best_efficientnetv2_s_pigs.pt


In [26]:
from google.colab import files
files.download("best_efficientnetv2_s_pigs.pt")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Inference — Predykcje na test set

In [27]:
BASE_DIR = "data/multiview_pig_posture_recognition"
TEST_CSV = os.path.join(BASE_DIR, "test.csv")

test = pd.read_csv(TEST_CSV)
test["source"]      = "test"
test["bbox_parsed"] = test["bbox"].apply(ast.literal_eval)

print("Test rows:", len(test))

Test rows: 11708


In [28]:
BATCH_SIZE = 32

test_dataset = PigPostureDataset(test, transform=val_transform, is_test=True)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=True,
)

In [29]:
row_ids     = []
predictions = []

model.eval()

with torch.no_grad():
    for images, ids in tqdm(test_loader, desc="Inference"):
        images = images.to(device, non_blocking=True)
        outputs = model(images)
        preds   = outputs.argmax(dim=1)
        row_ids.extend(ids)
        predictions.extend(preds.cpu().numpy())

print("Predykcje gotowe:", len(predictions))

Inference:   0%|          | 0/366 [00:00<?, ?it/s]

Predykcje gotowe: 11708


In [30]:
submission = pd.DataFrame({
    "row_id":   row_ids,
    "class_id": predictions,
})

assert len(submission) == len(test), "Liczba predykcji nie zgadza się z testem!"
assert set(submission["class_id"].unique()).issubset({0, 1, 2, 3, 4}), "Nieprawidłowe klasy!"

submission.to_csv("T2_efficientnetv2_wersja2.csv", index=False)
print("Zapisano: T2_efficientnetv2_wersja2.csv")
print(submission["class_id"].value_counts().sort_index())
submission.head()

Zapisano: T2_efficientnetv2_wersja2.csv
class_id
0    1483
1    1043
2     408
3    6415
4    2359
Name: count, dtype: int64


,row_id,class_id
0,test_pen1_tur_cam1_20250920_174649_0000,3
1,test_pen1_tur_cam1_20250920_174649_0001,1
2,test_pen1_tur_cam1_20250920_174649_0002,0
3,test_pen1_tur_cam1_20250920_174649_0003,0
4,test_pen1_tur_cam1_20250920_174649_0004,0


In [31]:
from google.colab import files
files.download("T2_efficientnetv2_wersja2.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>